# E-Commerce Lakehouse: Silver Customers

## Standardize customer fields

Remove surrounding spaces from customer identifiers and location
fields. Standardize cities to lowercase and states to uppercase.

Keep ZIP code prefixes as strings and pad numeric prefixes to five
characters. Preserve missing values for validation.

In [ ]:
from pyspark.sql.functions import (
    col, trim, lower, upper, lpad, when
)

bronze_customers_df = spark.table(
    "workspace.ecommerce.bronze_customers"
)

silver_customers_df = (
    bronze_customers_df
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_unique_id", trim(col("customer_unique_id")))
    .withColumn(
        "customer_zip_code_prefix",
        trim(col("customer_zip_code_prefix"))
    )
    .withColumn("customer_city", lower(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
)

# Pad only numeric prefixes of one to five digits.
# Leave other values unchanged so validation can identify them.
silver_customers_df = silver_customers_df.withColumn(
    "customer_zip_code_prefix",
    when(
        col("customer_zip_code_prefix").rlike(r"^[0-9]{1,5}$"),
        lpad(col("customer_zip_code_prefix"), 5, "0")
    ).otherwise(col("customer_zip_code_prefix"))
)

silver_customers_df.printSchema()

display(
    silver_customers_df.select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ).limit(5)
)

## Validate Silver customers

Check required fields, customer_id uniqueness, ZIP prefix format,
and valid Brazilian state codes. Confirm every Silver order links
to a customer.

customer_unique_id may repeat because a customer can make
multiple purchases; it is not required to be unique here.

In [ ]:
from pyspark.sql.functions import countDistinct

print("Bronze customer rows:", bronze_customers_df.count())
print("Silver customer rows:", silver_customers_df.count())

required_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

for column_name in required_columns:
    missing_count = silver_customers_df.filter(
        col(column_name).isNull()
        | (trim(col(column_name)) == "")
    ).count()

    print(f"Missing or blank {column_name}: {missing_count}")

duplicate_ids = (
    silver_customers_df
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate customer IDs:", duplicate_ids)

invalid_zip_count = silver_customers_df.filter(
    col("customer_zip_code_prefix").isNotNull()
    & ~col("customer_zip_code_prefix").rlike(r"^[0-9]{5}$")
).count()

print("Invalid non-null ZIP prefixes:", invalid_zip_count)

valid_states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

invalid_state_count = silver_customers_df.filter(
    col("customer_state").isNotNull()
    & ~col("customer_state").isin(valid_states)
).count()

print("Invalid non-null state codes:", invalid_state_count)

silver_orders_df = spark.table("workspace.ecommerce.silver_orders")

unmatched_orders = silver_orders_df.join(
    silver_customers_df.select("customer_id"),
    on="customer_id",
    how="left_anti"
).count()

print("Orders without a matching customer:", unmatched_orders)

silver_customers_df.agg(
    countDistinct("customer_unique_id").alias("distinct_customers")
).show()

## Refresh and verify Silver customers

Check the expected record count, required customer identifiers,
and customer_id uniqueness before replacing the Silver table.

Verify the saved record count and distinct customer count.
customer_unique_id may repeat across purchases.

In [ ]:
from pyspark.sql import functions as F

checks = silver_customers_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("customer_id").alias("distinct_ids"),
    F.countDistinct("customer_unique_id").alias("distinct_customers"),
    F.count(
        F.when(
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
            | F.col("customer_unique_id").isNull()
            | (F.trim(F.col("customer_unique_id")) == ""),
            1
        )
    ).alias("missing_identifiers")
).first()

if (
    checks["rows"] != 99441
    or checks["distinct_ids"] != checks["rows"]
    or checks["distinct_customers"] != 96096
    or checks["missing_identifiers"] != 0
):
    raise ValueError(
        "Silver customer validation failed. "
        "Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_customers"

(
    silver_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_customers_df = spark.table(table_name)

saved_checks = saved_customers_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("customer_unique_id").alias("distinct_customers")
).first()

if (
    saved_checks["rows"] != checks["rows"]
    or saved_checks["distinct_customers"] != checks["distinct_customers"]
):
    raise ValueError("Saved Silver customer counts do not match.")

print("Silver customers refreshed:", saved_checks["rows"])
print("Distinct customers:", saved_checks["distinct_customers"])